# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [9]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [10]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [11]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [12]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [13]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [14]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [15]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
I need to delegate the task of continuously monitoring and analyzing market data for AAPL to the Data Analyst to generate insights and alerts about significant market opportunities or threats. I also need to ask the Trading Strategy Developer to use statistical modeling and machine learning to identify trends and predict market movements for AAPL.

Action: Delegate work to co-worker
Action Input: {
    "coworker": "Data Analyst",
    "task": "Continuously monitor and analyze market data for AAPL",
    "context": "The task involves analyzing market data for AAPL using statistical methods to generate insights and alerts about significant market opportunities or threats."
}


> Entering new CrewAgentExecutor chain...

Final Answer: Apple Inc. (AAPL) Stock Price, News, Quote & History - Yahoo Finance Oops, something went wrong Skip to navigation Skip to main content Skip to right column News Today's news US Politics World Weather Climate change Health Wellness Mental health Sexual health Dermatology Oral health Hair loss Foot health Nutrition Healthy eating Meal delivery Weight loss Vitamins and supplements Fitness Equipment Exercise Women’s health Sleep Healthy aging Hearing Mobility Science Originals The 360 Newsletters Games Life Health Wellness Nutrition Fitness Healthy aging Mental health Sleep Your body Dermatology Children's health Foot health Hair loss Hearing Oral health Sexual health Women’s health Conditions Cardiovascular health Digestive health Endocrine system Parenting Family health So mini ways Style and beauty It Figures Unapologetically Horoscopes Shopping Style Accessories Clothing Luggage Shoes Beauty Hair Makeup Skincare Sunscreen Health Dental Fitness Hair loss Hearing aids Ment

Final Answer: 

AAPL Apple Inc. 246.06 +0.23% Summary News Research Chart Community Statistics Historical Data Profile Financials Analysis Options Holders Sustainability NasdaqGS - Nasdaq Real Time Price • USD Apple Inc. (AAPL) Follow Analyze with AI Time to act on AAPL? 245.50 +7.62 +(3.20%) At close: September 19 at 4:00:02 PM EDT 246.06 +0.56 +(0.23%) Overnight: 10:20:11 PM EDT This price reflects trading activity during the overnight session on the Blue Ocean ATS, available 8 PM to 4 AM ET, Sunday through Thursday, when regular markets are closed. Time to act on AAPL? Chart Range Bar 1D 5D 4.88% 1M 6.48% 6M 14.06% YTD -1.96% 1Y 7.27% 5Y 129.78% All 191,176.86% Key Events Mountain Advanced Chart Loading chart for AAPL Chart does not reflect overnight price. Previous Close 237.88 Open 241.18 Bid 240.00 x 100 Ask 258.21 x 100 Day's Range 240.21 - 246.30 52 Week Range 169.21 - 260.10 Volume 163,741,314 Avg. Volume 56,212,347 Market Cap (intraday) 3.643T Beta (5Y Monthly) 1.11 PE Ratio 

I need to delegate tasks to the appropriate team members and gather insights to develop trading strategies for AAPL.

Action: Delegate work to co-worker
Action Input: {"coworker": "Data Analyst", "task": "Provide insights on AAPL trading trends based on recent data analysis", "context": "I need detailed information on recent trading trends of AAPL to develop potential trading strategies"}


> Entering new CrewAgentExecutor chain...
I need to gather recent data on AAPL trading trends to provide detailed insights for potential trading strategies.

Action: Read website content
Action Input: {"website_url": "https://finance.yahoo.com/quote/AAPL"} 

Apple Inc. (AAPL) Stock Price, News, Quote & History - Yahoo Finance Oops, something went wrong Skip to navigation Skip to main content Skip to right column News Today's news US Politics World Weather Climate change Health Wellness Mental health Sexual health Dermatology Oral health Hair loss Foot health Nutrition Healthy eating Meal delivery We

Final Answer: 
Based on recent data analysis from Yahoo Finance on AAPL, the stock price of Apple Inc. (AAPL) is currently at 246.06, with a 0.23% increase. The stock has shown a range of 240.21 - 246.30 during the day, with a 52-week range of 169.21 - 260.10. The market cap is 3.643T, and the company's financial highlights show a profit margin of 24.30% and a return on equity of 149.81%. Additionally, the company offers various subscription-based services like Apple Arcade, Apple Fitness+, Apple Music, and Apple News+. The performance overview of AAPL shows a YTD return of 1.62% and a 1-year return of 7.77%. Considering these trends and financial indicators, potential trading strategies can be developed for AAPL.

> Finished chain.
 

Based on recent data analysis from Yahoo Finance on AAPL, the stock price of Apple Inc. (AAPL) is currently at 246.06, with a 0.23% increase. The stock has shown a range of 240.21 - 246.30 during the day, with a 52-week range of 169.21 - 260.10. The mark

Final Answer: Based on the insights gathered, I would recommend considering the following trading strategies for AAPL:

1. Stock Price Range: AAPL has a 52-week range of $169.21 to $260.10. Given the current price and volatility within this range, it might be beneficial to consider swing trading or trend following strategies to capitalize on price movements within this range.

2. Profit Margin: AAPL has a profit margin of 24.30%. This indicates a healthy level of profitability, which can be a positive factor for long-term investment strategies. Considering AAPL's consistent profitability, investors might consider a buy-and-hold strategy to benefit from potential future growth.

3. Return on Equity: AAPL has a return on equity of 149.81%. This high return on equity suggests that AAPL efficiently utilizes its equity to generate profits. Investors seeking high returns might find AAPL attractive for short to medium-term trading strategies.

4. Introduction of New Services: With the introdu

Final Answer: 
After conducting a comprehensive risk analysis for AAPL, here are some key insights:

- AAPL's stock price range has shown significant volatility in the past year, with a 1-year return of 7.77% and a 5-year return of 136.23%
- The company's profit margin is at 24.30%, indicating healthy profitability
- AAPL's return on equity (ROE) stands at an impressive 149.81%, reflecting strong performance in utilizing shareholder equity
- The introduction of new services such as Apple Arcade, Apple Fitness+, and Apple TV+ diversifies the revenue streams and enhances the company's competitive position in the market

Based on this analysis, it is important to consider the volatility in stock price, the profitability metrics, and the impact of new services on AAPL's overall risk exposure in trading activities.

> Finished chain.
 

After conducting a comprehensive risk analysis for AAPL, here are some key insights:

- AAPL's stock price range has shown significant volatility in the pas

- Display the final result as Markdown.

In [16]:
from IPython.display import Markdown
Markdown(result)

Risk Analysis Report for AAPL:

Introduction:
Apple Inc. (AAPL) is a leading technology company known for its innovative products and services. As part of our analysis, we have evaluated the proposed trading strategies for AAPL based on the following key risk factors:

1. Stock Price Volatility:
AAPL has experienced significant volatility in its stock price over the past year, with a 52-week range of $169.21 to $260.10. This volatility presents a risk for traders, as sudden price movements can lead to unexpected losses. To mitigate this risk, traders should consider implementing stop-loss orders and closely monitoring market trends to make informed decisions.

2. Profit Margin Risk:
AAPL's profit margin currently stands at 24.30%, indicating healthy profitability. However, fluctuations in profit margins can impact the company's financial performance and potentially affect its stock price. Traders should be aware of any changes in profit margins and adjust their trading strategies accordingly to minimize risks.

3. Return on Equity (ROE) Risk:
AAPL has a high return on equity of 149.81%, reflecting efficient use of shareholder equity to generate profits. While a high ROE is generally positive, it also increases the expectations for future performance. Traders should closely monitor AAPL's financial reports and market trends to assess any potential risks associated with changes in ROE.

4. New Services Introduction Risk:
AAPL's introduction of new services like Apple Arcade, Apple Fitness+, and Apple Music diversifies its revenue streams and enhances its market position. While this diversification can lead to revenue growth opportunities, it also introduces new risks related to market acceptance and competition. Traders should assess the impact of new services on AAPL's overall business strategy and adjust their trading plans accordingly.

Recommendations:
Based on the risk analysis conducted, we recommend the following mitigation strategies for traders considering AAPL:

1. Implement risk management tools such as stop-loss orders to protect against sudden price movements and minimize losses.
2. Monitor AAPL's financial reports and key performance indicators to stay informed about changes in profit margins and ROE.
3. Stay updated on market trends and competitor activities to assess the impact of new services on AAPL's market position.
4. Diversify trading strategies to include a combination of swing trading, trend following, buy-and-hold, and growth-oriented strategies to maximize trading efficiency and adaptability.

Overall, by understanding and addressing the risks associated with the proposed trading strategies for AAPL, traders can make informed decisions and optimize their trading performance in the dynamic market environment.